# ==============================================================================
# IMPORTS & SETUP
# ==============================================================================

In [ ]:
import os
import time
import logging
from pathlib import Path
from typing import List, Dict, Optional

import pandas as pd
from google.colab import userdata, drive, files
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, GoogleAPIError


In [ ]:

# Configure logging to track the experiment (can be saved to a file later)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Grouping variables here makes it easy to change experiment parameters later

In [ ]:
CONFIG = {
    "model_name": "gemini-1.5-flash",
    "data_dir": "/content/drive/My Drive/NECATRİX(NEC)", # Change to your target folder
    "output_csv": "parasite_classification_results.csv",
    "max_samples": 650,
    "max_retries": 6,
    "base_sleep_time": 2, # seconds
    "prompt": "This image belongs to a parasite, can you identify which class of parasite it belongs to? Answer in one word only."
}

# ==============================================================================
# CORE FUNCTIONS
# ==============================================================================

In [ ]:
def mount_and_authenticate() -> None:
    """Mounts Google Drive and authenticates the Gemini API."""
    logging.info("Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)
    
    logging.info("Authenticating Gemini API...")
    api_key = userdata.get('GOOGLE_API_KEY')
    if not api_key:
        raise ValueError("GOOGLE_API_KEY not found in Colab userdata.")
    genai.configure(api_key=api_key)

def classify_image(
    image_path: Path, 
    model: genai.GenerativeModel, 
    prompt: str
) -> Optional[str]:
    """Uploads an image, gets a classification, and cleans up the file."""
    sample_file = None
    try:
        # 1. Upload
        sample_file = genai.upload_file(path=str(image_path), display_name=image_path.name)
        
        # 2. Predict
        response = model.generate_content([sample_file, prompt])
        classification = response.text.strip()
        return classification

    except Exception as e:
        # Pass the exception up to the retry logic
        raise e

    finally:
        # 3. Cleanup: CRITICAL for large datasets to avoid storage quota limits
        if sample_file:
            try:
                genai.delete_file(sample_file.name)
            except Exception as cleanup_error:
                logging.warning(f"Failed to delete {sample_file.name} from Gemini: {cleanup_error}")



In [ ]:
def run_classification_pipeline(config: dict) -> pd.DataFrame:
    """Iterates through the image folder with exponential backoff and returns a DataFrame."""
    results: List[Dict[str, str]] = []
    folder_path = Path(config["data_dir"])
    
    if not folder_path.exists():
        raise FileNotFoundError(f"Directory not found: {folder_path}")

    # Gather valid images
    valid_extensions = {'.png', '.jpg', '.jpeg'}
    image_files = [f for f in folder_path.iterdir() if f.suffix.lower() in valid_extensions]
    
    logging.info(f"Found {len(image_files)} images in {folder_path}. Processing up to {config['max_samples']}...")

    model = genai.GenerativeModel(model_name=config["model_name"])

    for index, image_path in enumerate(image_files):
        if index >= config["max_samples"]:
            logging.info("Reached maximum sample limit.")
            break

        retries = 0
        success = False
        
        while retries < config["max_retries"] and not success:
            try:
                classification = classify_image(image_path, model, config["prompt"])
                
                results.append({
                    "filename": image_path.name,
                    "classification": classification,
                    "status": "Success",
                    "full_path": str(image_path)
                })
                
                logging.info(f"[{index+1}/{config['max_samples']}] {image_path.name} -> {classification}")
                success = True
                
                # Small sleep to respect API rate limits during standard loops
                time.sleep(1) 

            except Exception as e:
                wait_time = config["base_sleep_time"] ** retries
                logging.error(f"Error on {image_path.name}: {type(e).__name__} - {e}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
                retries += 1

        if not success:
            logging.error(f"Failed to process {image_path.name} after {config['max_retries']} retries.")
            results.append({
                "filename": image_path.name,
                "classification": "ERROR",
                "status": "Failed",
                "full_path": str(image_path)
            })

    # Convert results to a pandas DataFrame for structured analysis
    return pd.DataFrame(results)

# ==============================================================================
# EXECUTION
# ==============================================================================

In [ ]:
if __name__ == "__main__":
    try:
        mount_and_authenticate()
        
        # Run the pipeline
        df_results = run_classification_pipeline(CONFIG)
        
        # Save to CSV using pandas
        df_results.to_csv(CONFIG["output_csv"], index=False)
        logging.info(f"Results successfully saved to {CONFIG['output_csv']}")
        
        # Trigger download
        files.download(CONFIG["output_csv"])
        
    except Exception as e:
        logging.critical(f"Pipeline failed: {e}")